### Scott 10K regression

Task: Generate a function that:
1. Takes neuromaps values
2. Demeans them

3. Takes a scan row and removes everything APART FROM ROI values anda DATA_KEY
4. Takes ROI values and uses them in a linear regression
5. Extracts beta coefficients
6. Appends them and the DATA_KEY to a CSV

ALSO...

7. Generates null distribution of the patient data (MSR)
8. Uses the null dist to generate:
(a). adjusted r squared
(b). regression p value
9. Appends regression adjusted r squared and p value to a csv along with DATA_KEY

In [2]:
%%bash

mkdir -p /rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_regression

In [1]:
%%file /rds/general/user/sk4724/home/linear_spatial_regression_parallel_2.py

import pandas as pd
import numpy as np
import sklearn
from sklearn.linear_model import LinearRegression
import os, pickle,tqdm, neuromaps, netneurotools, nilearn, scipy
from nilearn import datasets, plotting, maskers
from scipy import ndimage, spatial
import nibabel as nib
from tqdm import tqdm
from neuromaps.nulls import moran
from netneurotools import utils
import sys

def get_r_sq(X,y,model):
    model.fit(X, y)
    yhat = model.predict(X)
    SS_Residual = sum((y - yhat) ** 2)
    SS_Total = sum((y - np.mean(y)) ** 2)
    r_squared = 1 - (SS_Residual / SS_Total)
    return r_squared
    
def get_adj_r_sq(X, y, model):
    """
    NOTE FROM SANKEITH: THIS IS NOT MINE.
    I TOOK THIS CODE FROM JUSTINE HANSEN THANK YOU JUSTINE HANSEN
    """
    r_squared = get_r_sq(X, y, model)
    adjusted_r_squared = 1 - (1 - r_squared) * (len(y) - 1) / (len(y) - X.shape[1] - 1)
    return adjusted_r_squared

def get_atlas_path(atlas):
    atlas_root = '/rds/general/project/c3nl_scott_students/live/sankeith/standards'
    if atlas.lower() == 'desikan':
        atlas_path = os.path.join(atlas_root, 'gm_only_MNI152_1mm_desikan+aseg.nii.gz')
    elif atlas.lower() == 'destrieux':
        atlas_path = os.path.join(atlas_root, 'gm_only_MNI152_1mm_destrieux+aseg.nii.gz')
    else:
        print('Please enter values of either Desikan or Destrieux')
        atlas_input = input('Provide an atlas value: ')
        atlas_path = get_atlas_path(atlas_input)
    return atlas_path

def get_centroids(img, labels=None, image_space=False):
    """
NOTE FROM SANKEITH: THIS IS NOT MINE I ADAPTED THIS CODE FROM AN OLD VERSION OF NETNEUROTOOLS
    
    Find centroids of `labels` in `img`.

    Parameters
    ----------
    img : niimg-like object
        3D image containing integer label at each point
    labels : array_like, optional
        List of labels for which to find centroids. If not specified all
        labels present in `img` will be used. Zero will be ignored as it is
        considered "background." Default: None
    image_space : bool, optional
        Whether to return xyz (image space) coordinates for centroids based
        on transformation in `img.affine`. Default: False

    Returns
    -------
    centroids : (N, 3) np.ndarray
        Coordinates of centroids for ROIs in input data
    """ 
    from nilearn.image import load_img, check_niimg_3d

    img = check_niimg_3d(img)
    data = nilearn.image.get_data(img)
    if labels is None:
        labels = np.trim_zeros(np.unique(data))
    centroids = np.vstack(ndimage.center_of_mass(data, labels=data,
                                                    index=labels))
    if image_space:
        centroids = nib.affines.apply_affine(img.affine, centroids)
    return centroids

def get_distmat(atlas):
    """
    Generate distance matrix for the ROIs. 
    The distance matrix essentially indicates relative distances of ROIs to each other.
    The distance matrix is n x n big (n = number of ROIs in an atlas)
    """
    atlas_path = get_atlas_path(atlas)
    gm_coords = get_centroids(atlas_path)
    distmat = scipy.spatial.distance_matrix(gm_coords, gm_coords)
    return distmat

def get_regr_p_val_moran(X, y, atlas, distmat, parcellation = None, n_perm = 10000):
    
    ## Get the original r squared
    emp_r_squared = get_r_sq(X, y, LinearRegression(fit_intercept = False))

    ## Selects atlas
    atlas_root = '/rds/general/project/c3nl_scott_students/live/sankeith/standards'
    atlas_path = get_atlas_path(atlas)
    if os.path.exists(atlas_path):
        
        #Generate distance matrix for the ROIs. 
        #The distance matrix essentially indicates relative distances of ROIs to each other.
        #The distance matrix is n x n big (n = number of ROIs in an atlas)
        # gm_coords = get_centroids(atlas_path)
        # distmat = scipy.spatial.distance_matrix(gm_coords, gm_coords)
        
        #Nulls of y are generated here
        gm_atlas_img = nilearn.image.load_img(atlas_path)
        labels_masker = maskers.NiftiLabelsMasker(labels_img=gm_atlas_img, standardize=None)
        labels_masker.fit()
        stat_img = labels_masker.inverse_transform(y.reshape(1,-1))
        stat_img.to_filename(f'/rds/general/project/c3nl_scott_students/ephemeral/sankeith/for_nulls.nii.gz')
        
        null_y_maps = neuromaps.nulls.moran(data= y.tolist(),
                                            distmat = distmat, # The docs say 'Providing this will cause atlas, density, and parcellation to be ignored.', but for some reason this scipt works with all of them combined idk why exactly
                                            n_perm = n_perm,
                                            atlas='mni152', 
                                            density='1mm', 
                                            parcellation = atlas_path,
                                            seed = 42)
        
        null_y_maps_transposed = np.transpose(null_y_maps)
        null_vals = [get_r_sq(X, null_y.reshape(-1, 1), LinearRegression(fit_intercept=False)) for null_y in null_y_maps_transposed]
    
        pval = (1 + np.sum(null_vals > emp_r_squared)) / (len(null_vals) + 1)
        return pval, emp_r_squared

    else:
        raise FileNotFoundError
    
def linear_spatial_regression(neuromaps_df_path,
                              scan_df_path,
                              regr_stats_path_root = '/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping',
                              coeff_df_path_root = '/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping',**kwargs):
    """
    This function aims to:

    * Conduct a linear regression between a scan's ROI values and corresponding ones from selected neurotransmitter maps
    * Find the scan's r squared, adjusted r squared and p value
    * Spit out the coefficients into one dataframe and the other stats into another
    """
    
    print(f"Starting linear spatial regression process:"
         f"\nNeuromaps DataFrame = {neuromaps_df_path.split('/')[-1]}"
         f"\nScan DataFrame = {scan_df_path.split('/')[-1]}")
    X_list = []
    neuromaps_df = pd.read_csv(neuromaps_df_path, low_memory = False)
    roi_column_order = neuromaps_df.iloc[:,0].tolist()
    neuromaps_df = neuromaps_df.loc[:, ~neuromaps_df.columns.str.contains('^Unnamed')]
    print(f"Demeaning neuromaps columns...")
    for col in neuromaps_df.columns:
        neuromaps_df_means = neuromaps_df.mean()
    demeaned_neuromaps_df = neuromaps_df - neuromaps_df_means
    X = demeaned_neuromaps_df.to_numpy()
    print(X.shape)
    
    scan_df = pd.read_csv(scan_df_path, low_memory = False)
    scan_df_cols = [x for x in scan_df.columns.tolist()]
    scan_roi_cols = scan_df_cols
    scan_roi_cols.remove('DATA_KEY')
    assert scan_roi_cols == roi_column_order

    basename = neuromaps_df_path.split('/')[-1].replace('.csv','')
    print(basename)

    coeff_prefix_dict = {'gm_only_transposed_desikan_neuromaps':'desikan',
                        'gm_only_transposed_destrieux_neuromaps':'destrieux',
    'gm_only_transposed_flipped_desikan_neuromaps':'flipped_desikan',
    'gm_only_transposed_flipped_destrieux_neuromaps':'flipped_destrieux'}

    coeff_prefix = coeff_prefix_dict[basename]

    basename_parts = basename.split('_')

    if 'desikan' in basename_parts:
        atlas = 'desikan'
    elif 'destrieux' in basename_parts:
        atlas = 'destrieux'

    distmat = get_distmat(atlas)
    print(f"Distance matrix for {atlas.lower()} calculated of size {distmat.shape}")

    coeff_rows = []
    stats_rows = []

    for index, row in tqdm(scan_df.iterrows(), total = scan_df.shape[0]):
        data_key = row['DATA_KEY']
        roi_rows = row.drop(labels = ['DATA_KEY']).transpose()
        y = roi_rows.to_numpy().reshape(-1,1)

        model = LinearRegression(fit_intercept=False)
        model.fit(X,y)
        r_squared = get_r_sq(X, y, model)
        adj_r_squared = get_adj_r_sq(X, y, model)

        pval,_ = get_regr_p_val_moran(X,y, atlas = atlas, distmat = distmat)
        
        beta_coeffs = model.coef_.tolist()
        
        coeff_row = {
            "DATA_KEY" : [data_key],
            f"{coeff_prefix}_D1" : [float(beta_coeffs[0][0])],
            f"{coeff_prefix}_D2" : [float(beta_coeffs[0][1])],
            f"{coeff_prefix}_DAT" : [float(beta_coeffs[0][2])],
            f"{coeff_prefix}_NET" : [float(beta_coeffs[0][3])],
            f"{coeff_prefix}_5HT1A" : [float(beta_coeffs[0][4])],
            f"{coeff_prefix}_5HT1B" : [float(beta_coeffs[0][5])],
            f"{coeff_prefix}_5HT2A" : [float(beta_coeffs[0][6])],
            f"{coeff_prefix}_5HT4" : [float(beta_coeffs[0][7])],
            f"{coeff_prefix}_5HT6" : [float(beta_coeffs[0][8])],
            f"{coeff_prefix}_5HTT" : [float(beta_coeffs[0][9])],
            f"{coeff_prefix}_a4b2" : [float(beta_coeffs[0][10])],
            f"{coeff_prefix}_M1" : [float(beta_coeffs[0][11])],
            f"{coeff_prefix}_vAChT" : [float(beta_coeffs[0][12])],
            f"{coeff_prefix}_NMDA" : [float(beta_coeffs[0][13])],
            f"{coeff_prefix}_mGluR5" : [float(beta_coeffs[0][14])],
            f"{coeff_prefix}_GABAA/BZ" : [float(beta_coeffs[0][15])],
            f"{coeff_prefix}_H3" : [float(beta_coeffs[0][16])],
            f"{coeff_prefix}_CB1" : [float(beta_coeffs[0][17])],
            f"{coeff_prefix}_MOR" : [float(beta_coeffs[0][18])]
        }
        coeff_rows.append(pd.DataFrame(coeff_row))

        stats_row = {
            "DATA_KEY" : [data_key],
            f"R-Squared" : [float(r_squared[0])],
            f"Adjusted R-Squared" : [float(adj_r_squared[0])],
            f"p-value" : [float(pval)]
        }

        stats_rows.append(pd.DataFrame(stats_row))
    
    coeff_df = pd.concat(coeff_rows, ignore_index = True)
    stats_df = pd.concat(stats_rows, ignore_index = True)
    print(coeff_df.shape)
    print(stats_df.shape)

    coeff_df_path = os.path.join(coeff_df_path_root,f"{coeff_prefix}_coeffs.csv")
    print(f"\nSaving coefficients to {coeff_df_path}")
    coeff_df.to_csv(coeff_df_path, index = False)

    stats_df_path = os.path.join(regr_stats_path_root,f"{coeff_prefix}_stats.csv")
    print(f"\nSaving linear regression stats to {stats_df_path}")
    stats_df.to_csv(stats_df_path, index = False)


arguments = [('/rds/general/project/c3nl_scott_students/live/sankeith/regression_generator/gm_only_transposed_desikan_neuromaps.csv','/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/gm_only_scraped_desikan_stats.csv'),
            ('/rds/general/project/c3nl_scott_students/live/sankeith/regression_generator/gm_only_transposed_destrieux_neuromaps.csv','/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/gm_only_scraped_destrieux_stats.csv'),
            ('/rds/general/project/c3nl_scott_students/live/sankeith/regression_generator/gm_only_transposed_flipped_desikan_neuromaps.csv','/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/gm_only_scraped_desikan_stats.csv'),
            ('/rds/general/project/c3nl_scott_students/live/sankeith/regression_generator/gm_only_transposed_flipped_destrieux_neuromaps.csv','/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/gm_only_scraped_destrieux_stats.csv'),]
    

argument_subset = arguments[int(sys.argv[1]) - 1]

linear_spatial_regression(neuromaps_df_path = argument_subset[0], 
                          scan_df_path = argument_subset[1])

Overwriting /rds/general/user/sk4724/home/linear_spatial_regression_parallel_2.py


In [1]:
%%file ~/submit_linear_regression.sh

#!/bin/bash
#PBS -l walltime=72:00:00
#PBS -l select=1:ncpus=1:mem=30gb
#PBS -N submit_linear_spatial_regression
#PBS -J 1-4
#PBS -o /rds/general/user/sk4724/home/
#PBS -e /rds/general/user/sk4724/home/

source /rds/general/user/sk4724/home/venv_1/bin/activate

chmod -Rf 775 /rds/general/user/sk4724/home/linear_spatial_regression_parallel_2.py
chmod -Rf 775 /rds/general/user/sk4724/home/submit_linear_regression.sh

python /rds/general/user/sk4724/home/linear_spatial_regression_parallel_2.py ${PBS_ARRAY_INDEX}

Overwriting /rds/general/user/sk4724/home/submit_linear_regression.sh


In [2]:
%%bash

chmod -Rf 775 /rds/general/user/sk4724/home/submit_linear_regression.sh
qsub /rds/general/user/sk4724/home/submit_linear_regression.sh

2178235[].pbs-7


#### Perform an rmANOVA on r squareds, adj r squareds and p-values

In [5]:
import pandas as pd
import numpy as np
import os
import scipy
import statsmodels
from statsmodels.stats.anova import AnovaRM
import pingouin as pg

stats_root_path = '/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping'

def regr_stats_anova(var):

    f_desi = pd.read_csv(os.path.join(stats_root_path, 'flipped_desikan_stats.csv'), usecols=['DATA_KEY', var]).rename(columns={var: 'desikan_flipped'})
    desi = pd.read_csv(os.path.join(stats_root_path, 'desikan_stats.csv'), usecols=['DATA_KEY', var]).rename(columns={var: 'desikan_original'})
    f_dest = pd.read_csv(os.path.join(stats_root_path, 'flipped_destrieux_stats.csv'), usecols=['DATA_KEY', var]).rename(columns={var: 'destrieux_flipped'})
    dest = pd.read_csv(os.path.join(stats_root_path, 'destrieux_stats.csv'), usecols=['DATA_KEY', var]).rename(columns={var: 'destrieux_original'})
                                                                                                                                
    assert f_desi.shape == desi.shape
    assert desi.shape == f_dest.shape
    assert f_dest.shape == dest.shape

    merged_df = f_desi.merge(desi, on='DATA_KEY').merge(f_dest, on='DATA_KEY').merge(dest, on='DATA_KEY')

    melted_df = pd.melt(merged_df, id_vars=['DATA_KEY'], var_name='method', value_name='metric_value')
    melted_df[['atlas', 'orientation']] = melted_df['method'].str.split('_', expand=True)

    melted_df['DATA_KEY'] = melted_df['DATA_KEY'].astype(object)

    melted_df['metric_value'] = pd.to_numeric(melted_df['metric_value'], errors='coerce')

    aov = pg.rm_anova(data=melted_df, 
                      dv='metric_value', 
                      within=['atlas', 'orientation'], 
                      subject='DATA_KEY', 
                      detailed=True)

    print(aov)

    posthoc = pg.pairwise_tests(data=melted_df, 
                                dv='metric_value', 
                                within='method', 
                                subject='DATA_KEY', 
                                padjust='bonf')

    print(posthoc)

for i in [f"R-Squared", f"Adjusted R-Squared", f"p-value"]:
    print(f'Starting rm-ANOVA for {i}')
    regr_stats_anova(i)

Starting rm-ANOVA for R-Squared
                Source           SS  ddof1  ddof2           MS             F  \
0                atlas  1864.427680      1  15724  1864.427680  1.603783e+11   
1          orientation   103.496995      1  15724   103.496995  2.180158e+11   
2  atlas * orientation     0.613271      1  15724     0.613271  1.032946e+09   

   p_unc  p_GG_corr       ng2  eps  
0    0.0        0.0  0.999989  1.0  
1    0.0        0.0  0.999796  1.0  
2    0.0        0.0  0.966753  1.0  
  Contrast                  A                   B  Paired  Parametric  \
0   method    desikan_flipped    desikan_original    True        True   
1   method    desikan_flipped   destrieux_flipped    True        True   
2   method    desikan_flipped  destrieux_original    True        True   
3   method   desikan_original   destrieux_flipped    True        True   
4   method   desikan_original  destrieux_original    True        True   
5   method  destrieux_flipped  destrieux_original    True    

/rds/general/user/sk4724/home/venv_1/lib/python3.13/site-packages/pingouin/bayesian.py:140: RuntimeWarning: invalid value encountered in scalar divide
  bf10 = 1 / ((1 + t**2 / df) ** (-(df + 1) / 2) / integr)
/rds/general/user/sk4724/home/venv_1/lib/python3.13/site-packages/pingouin/bayesian.py:140: RuntimeWarning: invalid value encountered in scalar divide
  bf10 = 1 / ((1 + t**2 / df) ** (-(df + 1) / 2) / integr)
/rds/general/user/sk4724/home/venv_1/lib/python3.13/site-packages/pingouin/bayesian.py:140: RuntimeWarning: invalid value encountered in scalar divide
  bf10 = 1 / ((1 + t**2 / df) ** (-(df + 1) / 2) / integr)
/rds/general/user/sk4724/home/venv_1/lib/python3.13/site-packages/pingouin/bayesian.py:140: RuntimeWarning: invalid value encountered in scalar divide
  bf10 = 1 / ((1 + t**2 / df) ** (-(df + 1) / 2) / integr)
/rds/general/user/sk4724/home/venv_1/lib/python3.13/site-packages/pingouin/bayesian.py:140: RuntimeWarning: invalid value encountered in scalar divide
  bf10 = 

                Source            SS  ddof1  ddof2            MS  \
0                atlas  20279.365739      1  15724  20279.365739   
1          orientation    152.804758      1  15724    152.804758   
2  atlas * orientation      3.059612      1  15724      3.059612   

              F  p_unc  p_GG_corr       ng2  eps  
0  4.253302e+11    0.0        0.0  0.999998  1.0  
1  1.916248e+11    0.0        0.0  0.999793  1.0  
2  3.152289e+09    0.0        0.0  0.989756  1.0  
  Contrast                  A                   B  Paired  Parametric  \
0   method    desikan_flipped    desikan_original    True        True   
1   method    desikan_flipped   destrieux_flipped    True        True   
2   method    desikan_flipped  destrieux_original    True        True   
3   method   desikan_original   destrieux_flipped    True        True   
4   method   desikan_original  destrieux_original    True        True   
5   method  destrieux_flipped  destrieux_original    True        True   

           

/rds/general/user/sk4724/home/venv_1/lib/python3.13/site-packages/pingouin/bayesian.py:140: RuntimeWarning: invalid value encountered in scalar divide
  bf10 = 1 / ((1 + t**2 / df) ** (-(df + 1) / 2) / integr)
/rds/general/user/sk4724/home/venv_1/lib/python3.13/site-packages/pingouin/bayesian.py:140: RuntimeWarning: invalid value encountered in scalar divide
  bf10 = 1 / ((1 + t**2 / df) ** (-(df + 1) / 2) / integr)
/rds/general/user/sk4724/home/venv_1/lib/python3.13/site-packages/pingouin/bayesian.py:140: RuntimeWarning: invalid value encountered in scalar divide
  bf10 = 1 / ((1 + t**2 / df) ** (-(df + 1) / 2) / integr)
/rds/general/user/sk4724/home/venv_1/lib/python3.13/site-packages/pingouin/bayesian.py:140: RuntimeWarning: invalid value encountered in scalar divide
  bf10 = 1 / ((1 + t**2 / df) ** (-(df + 1) / 2) / integr)
/rds/general/user/sk4724/home/venv_1/lib/python3.13/site-packages/pingouin/bayesian.py:140: RuntimeWarning: invalid value encountered in scalar divide
  bf10 = 

                Source            SS  ddof1  ddof2            MS        F  \
0                atlas  4.621162e-35      1  15724  4.621162e-35 -15724.0   
1          orientation  4.621162e-35      1  15724  4.621162e-35 -15724.0   
2  atlas * orientation -4.621162e-35      1  15724 -4.621162e-35 -15724.0   

   p_unc  p_GG_corr  ng2  eps  
0    1.0        1.0  1.0  1.0  
1    1.0        1.0  1.0  1.0  
2    1.0        1.0  1.0  1.0  
  Contrast                  A                   B  Paired  Parametric  \
0   method    desikan_flipped    desikan_original    True        True   
1   method    desikan_flipped   destrieux_flipped    True        True   
2   method    desikan_flipped  destrieux_original    True        True   
3   method   desikan_original   destrieux_flipped    True        True   
4   method   desikan_original  destrieux_original    True        True   
5   method  destrieux_flipped  destrieux_original    True        True   

       dof alternative p_adjust BF10  hedges  
0  1

/rds/general/user/sk4724/home/venv_1/lib/python3.13/site-packages/pingouin/parametric.py:245: UserWarning: x and y are equals. Cannot compute T or p-value.
  warnings.warn("x and y are equals. Cannot compute T or p-value.")
/rds/general/user/sk4724/home/venv_1/lib/python3.13/site-packages/pingouin/parametric.py:245: UserWarning: x and y are equals. Cannot compute T or p-value.
  warnings.warn("x and y are equals. Cannot compute T or p-value.")
/rds/general/user/sk4724/home/venv_1/lib/python3.13/site-packages/pingouin/parametric.py:245: UserWarning: x and y are equals. Cannot compute T or p-value.
  warnings.warn("x and y are equals. Cannot compute T or p-value.")
/rds/general/user/sk4724/home/venv_1/lib/python3.13/site-packages/pingouin/parametric.py:245: UserWarning: x and y are equals. Cannot compute T or p-value.
  warnings.warn("x and y are equals. Cannot compute T or p-value.")
/rds/general/user/sk4724/home/venv_1/lib/python3.13/site-packages/pingouin/parametric.py:245: UserWarnin

In [ ]:
import glob
import os
import subprocess

import nilearn
import nibabel as nib
import numpy as np
from scipy import ndimage
from sklearn.utils.validation import check_array

def get_centroids(img, labels=None, image_space=False):
    """
NOTE FROM SANKEITH: THIS IS NOT MINE I TOOK THIS CODE FROM AN OLD VERSION OF NETNEUROTOOLS
    
    Find centroids of `labels` in `img`.

    Parameters
    ----------
    img : niimg-like object
        3D image containing integer label at each point
    labels : array_like, optional
        List of labels for which to find centroids. If not specified all
        labels present in `img` will be used. Zero will be ignored as it is
        considered "background." Default: None
    image_space : bool, optional
        Whether to return xyz (image space) coordinates for centroids based
        on transformation in `img.affine`. Default: False

    Returns
    -------
    centroids : (N, 3) np.ndarray
        Coordinates of centroids for ROIs in input data
    """ 
    from nilearn.image import load_img, check_niimg_3d

    img = nib.load(img)
    data = img.get_fdata()

    if labels is None:
        labels = np.trim_zeros(np.unique(data))

    centroids = np.vstack(ndimage.center_of_mass(data, labels=data,
                                                    index=labels))

    if image_space:
        centroids = nib.affines.apply_affine(img.affine, centroids)

    return centroids

coords = get_centroids('/rds/general/project/c3nl_scott_students/live/sankeith/standards/gm_only_MNI152_1mm_desikan+aseg.nii.gz')
print(coords)

In [ ]:
import pandas as pd
import numpy as np
import sklearn
from sklearn.linear_model import LinearRegression
import os, pickle,tqdm, neuromaps, netneurotools, nilearn, scipy
from scipy import ndimage
import nibabel as nib
from tqdm import tqdm
from neuromaps.nulls import moran
from netneurotools import utils

def get_r_sq(X,y,model):
    model.fit(X, y)
    yhat = model.predict(X)
    SS_Residual = sum((y - yhat) ** 2)
    SS_Total = sum((y - np.mean(y)) ** 2)
    r_squared = 1 - (SS_Residual / SS_Total)
    return r_squared
    
def get_adj_r_sq(X, y, model):
    """
    NOTE FROM SANKEITH: THIS IS NOT MINE.
    I TOOK THIS CODE FROM JUSTINE HANSEN THANK YOU JUSTINE HANSEN
    """
    r_squared = get_r_sq(X, y, model)
    adjusted_r_squared = 1 - (1 - r_squared) * (len(y) - 1) / (len(y) - X.shape[1] - 1)
    return adjusted_r_squared

def get_atlas_path(atlas):
    atlas_root = '/rds/general/project/c3nl_scott_students/live/sankeith/standards'
    if atlas.lower() == 'desikan':
        atlas_path = os.path.join(atlas_root, 'gm_only_MNI152_1mm_desikan_aparc+aseg.nii.gz')
    elif atlas.lower() == 'destrieux':
        atlas_path = os.path.join(atlas_root, 'gm_only_MNI152_1mm_destrieux_aparc.a2009+aseg.nii.gz')
    return atlas_path

def get_centroids(img, labels=None, image_space=False):
    """
NOTE FROM SANKEITH: THIS IS NOT MINE I TOOK THIS CODE FROM AN OLD VERSION OF NETNEUROTOOLS
    
    Find centroids of `labels` in `img`.

    Parameters
    ----------
    img : niimg-like object
        3D image containing integer label at each point
    labels : array_like, optional
        List of labels for which to find centroids. If not specified all
        labels present in `img` will be used. Zero will be ignored as it is
        considered "background." Default: None
    image_space : bool, optional
        Whether to return xyz (image space) coordinates for centroids based
        on transformation in `img.affine`. Default: False

    Returns
    -------
    centroids : (N, 3) np.ndarray
        Coordinates of centroids for ROIs in input data
    """ 
    from nilearn.image import load_img, check_niimg_3d

    img = nilearn.image.load_img(img)
    img = check_niimg_3d(img)
    data = np.asarray(img.dataobj)

    if labels is None:
        labels = np.trim_zeros(np.unique(data))

    centroids = np.vstack(ndimage.center_of_mass(data, labels=data,
                                                    index=labels))

    if image_space:
        centroids = nib.affines.apply_affine(img.affine, centroids)

    return centroids
    
    

def get_regr_p_val_moran(X, y, atlas, n_perm = 10000):
    
    ## Get the original adjusted r squared
    emp_adj_r_squared = get_adj_r_sq(X, y, LinearRegression(fit_intercept = False))

    ## Selects atlas
    atlas_root = '/rds/general/project/c3nl_scott_students/live/sankeith/standards'
    try:
        atlas_path = get_atlas_path(atlas)
    except Exception:
        print('Please enter values of either Desikan or Destrieux')
        atlas_input = input('Provide an atlas value: ')
        atlas_path = get_atlas_path(atlas_input)
    if os.path.exists(atlas_path):
        #Generate distance matrix for the ROIs. 
        #The distance matrix essentially indicates relative distances of ROIs to each other.
        #The distance matrix is n x n big (n = number of ROIs in an atlas)
        gm_coords = get_centroids(atlas_path)
        print(gm_coords)
        
        #     ## Nulls of y are generated here
    #     null_y_maps = neuromaps.nulls.moran(data = y, atlas = atlas_path, n_perm = n_perm, seed = 42)
    #     print(f'Null y maps shape = {null_y_maps.shape}')
    # else:
    #     raise FileNotFoundError
    

foo = 'foo'
bar = 'bar'

def linear_spatial_regression(neuromaps_df_path,
                              scan_df_path,
                              regr_stats_path,
                              coeff_df_path_root = '/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping',**kwargs):
    
    print(f"Starting linear spatial regression process:"
         f"\nNeuromaps DataFrame = {neuromaps_df_path.split('/')[-1]}"
         f"\nScan DataFrame = {scan_df_path.split('/')[-1]}")
    X_list = []
    neuromaps_df = pd.read_csv(neuromaps_df_path, low_memory = False)
    roi_column_order = neuromaps_df.iloc[:,0].tolist()
    neuromaps_df = neuromaps_df.loc[:, ~neuromaps_df.columns.str.contains('^Unnamed')]
    print(f"Demeaning neuromaps columns...")
    for col in neuromaps_df.columns:
        neuromaps_df_means = neuromaps_df.mean()
    demeaned_neuromaps_df = neuromaps_df - neuromaps_df_means
    X = demeaned_neuromaps_df.to_numpy()
    print(X.shape)
    
    #     col_array = neuromaps_df[col].to_numpy().reshape(-1,1)
    #     demeaned_col_array = col_array - np.mean(col_array)
    #     print(demeaned_col_array)
    #     X_list.append(demeaned_col_array)
    # X = np.hstack(X_list)
    # print(X)
    # print(X.shape)
    # np.savetxt('/rds/general/project/c3nl_scott_students/ephemeral/sankeith/scott_10k_regression/X.txt', X, delimiter = ',')
    
    scan_df = pd.read_csv(scan_df_path, low_memory = False)
    scan_df_cols = [x for x in scan_df.columns.tolist()]
    scan_roi_cols = scan_df_cols
    scan_roi_cols.remove('DATA_KEY')
    assert scan_roi_cols == roi_column_order

    basename = neuromaps_df_path.split('/')[-1].replace('.csv','')
    print(basename)

    coeff_prefix_dict = {'gm_only_transposed_desikan_neuromaps':'desikan',
                        'gm_only_transposed_destrieux_neuromaps':'destrieux',
    'gm_only_transposed_flipped_desikan_neuromaps':'flipped_desikan',
    'gm_only_transposed_flipped_destrieux_neuromaps':'flipped_destrieux'}

    coeff_prefix = coeff_prefix_dict[basename]

    coeff_rows = []

    for index, row in tqdm(scan_df.iterrows(), total = scan_df.shape[0]):
        data_key = row['DATA_KEY']
        roi_rows = row.drop(labels = ['DATA_KEY']).transpose()
        y = roi_rows.to_numpy().reshape(-1,1)

        model = LinearRegression(fit_intercept=False)
        model.fit(X,y)
        r_squared = get_r_sq(X, y, model)
        adj_r_squared = get_adj_r_sq(X, y, model)

        basename_parts = basename.split('_')

        
        if 'desikan' in basename_parts:
            atlas = 'desikan'
        elif 'destrieux' in basename_parts:
            atlas = 'destrieux'

        get_regr_p_val_moran(X,y, atlas = atlas)
        
        beta_coeffs = model.coef_.tolist()
        
        coeff_row = {
            "DATA_KEY" : [data_key],
            f"{coeff_prefix}_D1" : [float(beta_coeffs[0][0])],
            f"{coeff_prefix}_D2" : [float(beta_coeffs[0][1])],
            f"{coeff_prefix}_DAT" : [float(beta_coeffs[0][2])],
            f"{coeff_prefix}_NET" : [float(beta_coeffs[0][3])],
            f"{coeff_prefix}_5HT1A" : [float(beta_coeffs[0][4])],
            f"{coeff_prefix}_5HT1B" : [float(beta_coeffs[0][5])],
            f"{coeff_prefix}_5HT2A" : [float(beta_coeffs[0][6])],
            f"{coeff_prefix}_5HT4" : [float(beta_coeffs[0][7])],
            f"{coeff_prefix}_5HT6" : [float(beta_coeffs[0][8])],
            f"{coeff_prefix}_5HTT" : [float(beta_coeffs[0][9])],
            f"{coeff_prefix}_a4b2" : [float(beta_coeffs[0][10])],
            f"{coeff_prefix}_M1" : [float(beta_coeffs[0][11])],
            f"{coeff_prefix}_vAChT" : [float(beta_coeffs[0][12])],
            f"{coeff_prefix}_NMDA" : [float(beta_coeffs[0][13])],
            f"{coeff_prefix}_mGluR5" : [float(beta_coeffs[0][14])],
            f"{coeff_prefix}_GABAA/BZ" : [float(beta_coeffs[0][15])],
            f"{coeff_prefix}_H3" : [float(beta_coeffs[0][16])],
            f"{coeff_prefix}_CB1" : [float(beta_coeffs[0][17])],
            f"{coeff_prefix}_MOR" : [float(beta_coeffs[0][18])]
        }
        coeff_rows.append(pd.DataFrame(coeff_row))

    coeff_df = pd.concat(coeff_rows, ignore_index = True)
    print(coeff_df.shape)

    coeff_df_path = os.path.join(coeff_df_path_root,f"{coeff_prefix}_coeffs.csv")
    print(f"\nSaving coefficients to {coeff_df_path}")
    coeff_df.to_csv(coeff_df_path, index = False)
            
    

linear_spatial_regression('/rds/general/project/c3nl_scott_students/live/sankeith/regression_generator/gm_only_transposed_desikan_neuromaps.csv', 
                          '/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/gm_only_scraped_desikan_stats.csv', foo)

In [37]:
import pandas as pd

df = pd.read_csv('/rds/general/project/c3nl_scott_students/live/sankeith/scott_10k_housekeeping/gm_only_scraped_desikan_stats.csv', low_memory = False)
df

,DATA_KEY,Left-Cerebellum-Cortex,Left-Thalamus-Proper,Left-Caudate,Left-Putamen,Left-Pallidum,Left-Hippocampus,Left-Amygdala,Left-Accumbens-area,Left-VentralDC,...,ctx-rh-rostralanteriorcingulate,ctx-rh-rostralmiddlefrontal,ctx-rh-superiorfrontal,ctx-rh-superiorparietal,ctx-rh-superiortemporal,ctx-rh-supramarginal,ctx-rh-frontalpole,ctx-rh-temporalpole,ctx-rh-transversetemporal,ctx-rh-insula
0,128_S_0216_ADNI-T1_2006-09-12_07_31_03.0,-0.120155,-0.094790,2.661165,0.433559,0.001389,-0.266798,-0.281049,1.300440,0.009493,...,0.510246,0.060275,0.290023,0.266682,0.067183,-0.020678,0.083241,-0.313796,-0.404684,0.644190
1,037_S_5162_ADNI-T1_2013-05-03_08_21_20.0,0.198382,-0.061896,-0.442138,-0.253383,-0.001225,-0.203326,-0.105643,-0.110128,0.016031,...,-0.563009,-0.408258,-0.206622,-0.033989,0.008879,-0.246507,-0.155639,-0.168328,0.356982,-0.516417
2,128_S_0216_ADNI-T1_2007-02-22_10_49_53.0,-0.124910,-0.152267,2.726367,0.469988,-0.000538,-0.110894,-0.501716,1.216734,0.014715,...,0.474148,0.059725,0.331007,0.349893,0.187689,0.008693,0.321693,-0.182294,0.118097,0.737351
3,128_S_0216_ADNI-T1_2008-03-12_14_25_20.0,-0.068215,-0.254262,2.722131,0.365798,-0.004132,-0.453145,-0.388340,0.853076,-0.035231,...,0.479822,-0.008867,0.197917,0.220735,0.091722,-0.017417,0.097337,-0.435361,-0.278039,0.571793
4,033_S_0513_ADNI-T1_2006-05-18_11_34_42.0,0.334040,-0.011875,0.124466,0.175821,-0.000346,0.091771,-0.525354,0.095162,0.067085,...,0.314189,0.282002,0.384681,0.085946,-0.026707,0.078651,0.253283,0.216642,-0.070612,0.191049
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
15721,126_S_4458_ADNI-T1_2013-02-05_14_17_06.0,0.467601,0.105351,-1.295222,-0.268118,-0.002946,-0.536611,-1.326829,-0.379822,0.063731,...,-0.198249,-0.291722,-0.178429,-0.284918,-0.582935,-0.165127,-0.193240,-0.479188,-0.289211,-0.517265
15722,126_S_4458_ADNI-T1_2012-01-20_09_24_18.0,-0.026939,-0.333835,0.053818,0.074471,0.265750,-0.253148,0.000000,0.589210,-0.221053,...,-0.821209,-0.192837,-0.182899,-0.061197,0.018697,0.000000,-0.070388,0.028917,-0.004826,0.019481
15723,123_S_4127_ADNI-T1_2019-10-18_13_09_23.0,-0.341604,0.022974,-0.810474,-0.145802,-0.001884,0.247182,0.133564,-0.703227,-0.011943,...,-0.954821,-0.091393,-0.033256,-0.045593,-0.349320,0.002760,-0.272219,-0.203989,-0.110345,-0.514851
15724,123_S_4127_ADNI-T1_2011-08-03_10_48_33.0,-0.071342,0.080242,-0.088182,0.221259,0.367241,0.153299,-0.016588,-0.026555,0.529316,...,0.251602,-0.154149,0.027477,-0.092725,-0.077213,-0.031725,-0.170597,-0.052008,-0.046949,0.029292
